# Benchmark results - tables

Reads and displays the aggregated CSVs in `benchmark_tapas/results/tables/` (produced by `analysis/aggregate.py`). Run all cells.

## 0. Locate + list the tables

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

# Locate benchmark_tapas/results/tables/ regardless of where the kernel started.
_here = Path.cwd()
_cands = [_here / "../results/tables", _here / "results/tables",
          _here / "benchmark_tapas/results/tables"]
TABLES = next((c.resolve() for c in _cands if c.exists()), None)
assert TABLES is not None, "results/tables/ not found -- run the run_*.py scripts + analysis/aggregate.py first."
print("Reading from:", TABLES)
print("CSVs:", [p.name for p in sorted(TABLES.glob("*.csv"))])

Reading from: /Users/PengOlivia/Desktop/Uni/USYD VRI/priv-sdg/benchmark_tapas/results/tables
CSVs: ['benchmark_privacy_per_attack.csv', 'summary_combined.csv', 'summary_privacy.csv', 'summary_utility.csv']


## 1. Privacy summary  (`summary_privacy.csv`)

Per method: worst-case & mean MIA AUC, mean membership advantage, # attacks that succeed, worst-case effective epsilon (95% CI), formal e, and the eff-eps gap.

In [ ]:
priv = pd.read_csv(TABLES / "summary_privacy.csv")
priv = pd.read_csv(TABLES / "summary_privacy.csv")
cols = ["label", "worst_case_auc", "mean_advantage", "n_attacks_succeed",
        "worst_case_eff_eps_low95", "formal_epsilon", "eff_eps_gap"]
display(priv[cols])

,method,label,dp,kind,n_attacks,worst_case_auc,worst_auc_attack,mean_auc,mean_advantage,n_attacks_succeed,worst_case_eff_eps_low95,worst_case_eff_eps_high95,n_pointwise_inf,formal_epsilon,eff_eps_gap,num_train,num_test
0,bayesian_network,BN (no-DP),False,statistical,5,1.0,Groundhog,0.4,0.4,2,2.209964,inf,2,NaN,NaN,50,100
1,privbayes,PrivBayes (DP),True,statistical,5,1.0,Groundhog,0.5,0.4,2,2.209964,inf,2,1.0,1.209964,50,100
2,ctgan,CTGAN (no-DP),False,neural,5,1.0,Groundhog,0.7,0.6,3,2.209964,inf,3,NaN,NaN,50,100
3,dpgan,DPGAN (DP),True,neural,5,1.0,Groundhog,0.7,0.6,3,2.209964,inf,3,1.0,1.209964,50,100


,label,worst_case_auc,mean_advantage,n_attacks_succeed,worst_case_eff_eps_low95,formal_epsilon,eff_eps_gap
0,BN (no-DP),1.0,0.4,2,2.209964,NaN,NaN
1,PrivBayes (DP),1.0,0.4,2,2.209964,1.0,1.209964
2,CTGAN (no-DP),1.0,0.6,3,2.209964,NaN,NaN
3,DPGAN (DP),1.0,0.6,3,2.209964,1.0,1.209964


## 2. Utility & fidelity summary  (`summary_utility.csv`)

TSTR (XGBoost / linear) + SDMetrics fidelity (KS/TV-complement, correlation & contingency similarity).

In [3]:
util = pd.read_csv(TABLES / "summary_utility.csv")
display(util.round(3))

,method,label,xgb_gt,xgb_syn_id,xgb_syn_ood,linear_model_syn_id,feat_rank_distance_corr,KSComplement,TVComplement,CorrelationSimilarity,ContingencySimilarity
0,bayesian_network,BN (no-DP),0.92,0.767,0.776,0.665,0.225,0.729,0.897,0.973,0.789
1,privbayes,PrivBayes (DP),0.92,0.883,0.887,0.781,0.614,0.936,0.947,0.969,0.877
2,ctgan,CTGAN (no-DP),0.92,0.872,0.872,0.746,0.444,0.706,0.852,0.985,0.728
3,dpgan,DPGAN (DP),0.92,0.394,0.395,0.757,-0.028,0.199,0.210,0.956,0.059


## 3. Combined tradeoff axes  (`summary_combined.csv`)

One row per method with the tradeoff-plot fields (utility vs mean advantage) + eff-eps.

In [4]:
comb = pd.read_csv(TABLES / "summary_combined.csv")
display(comb.round(3))

,method,label,dp,kind,worst_case_auc,mean_auc,mean_advantage,n_attacks_succeed,worst_case_eff_eps_low95,worst_case_eff_eps_high95,formal_epsilon,eff_eps_gap,xgb_syn_id,xgb_gt,linear_model_syn_id,feat_rank_distance_corr,KSComplement,TVComplement
0,bayesian_network,BN (no-DP),False,statistical,1.0,0.4,0.4,2,2.21,inf,NaN,NaN,0.767,0.92,0.665,0.225,0.729,0.897
1,privbayes,PrivBayes (DP),True,statistical,1.0,0.5,0.4,2,2.21,inf,1.0,1.21,0.883,0.92,0.781,0.614,0.936,0.947
2,ctgan,CTGAN (no-DP),False,neural,1.0,0.7,0.6,3,2.21,inf,NaN,NaN,0.872,0.92,0.746,0.444,0.706,0.852
3,dpgan,DPGAN (DP),True,neural,1.0,0.7,0.6,3,2.21,inf,1.0,1.21,0.394,0.92,0.757,-0.028,0.199,0.210


## 4. Per-attack MIA results  (`benchmark_privacy_per_attack.csv`)

All 5 attacks x 4 methods (full table), then AUC and membership-advantage pivoted for readability.

In [5]:
per_attack = pd.read_csv(TABLES / "benchmark_privacy_per_attack.csv")
per_attack["attack_short"] = per_attack["attack"].str.split("(").str[0]

# full per-attack table
display(per_attack.round(3))

# readable pivots: AUC and membership advantage, attack x method
ATTACKS = ["Groundhog", "ShadowModelling", "LocalNeighbourhood",
           "ProbabilityEstimation", "ClosestDistance"]
METHODS = ["bayesian_network", "privbayes", "ctgan", "dpgan"]
for metric, title in [("auc", "MIA AUC"), ("mia_advantage", "Membership advantage (tp - fp)")]:
    piv = (per_attack.pivot_table(index="attack_short", columns="method", values=metric)
                     .reindex(index=ATTACKS, columns=METHODS))
    display(Markdown(f"**{title} - attack x method:**"))
    display(piv.round(2))

,attack,num_train,num_test,wall_time_s,peak_memory_mb,auc,mia_advantage,privacy_gain,tp,fp,eff_epsilon_pointwise,eff_epsilon_pointwise_is_inf,eps_low_90,eps_high_90,eps_low_95,eps_high_95,eps_low_99,eps_high_99,method,dp,kind,formal_epsilon,attack_short
0,Groundhog,50,100,182.56,358.99,1.0,1.0,0.0,1.0,0.0,NaN,True,2.39,inf,2.21,inf,1.878,inf,bayesian_network,False,statistical,NaN,Groundhog
1,ShadowModelling(RandomQueries),50,100,72.52,1.01,1.0,1.0,0.0,1.0,0.0,NaN,True,2.39,inf,2.21,inf,1.878,inf,bayesian_network,False,statistical,NaN,ShadowModelling
2,"LocalNeighbourhood(L_2, 0.250411558405805, acc...",50,100,0.67,1.09,0.0,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,bayesian_network,False,statistical,NaN,LocalNeighbourhood
3,ProbabilityEstimation(KernelDensity()),50,100,0.57,0.94,0.0,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,bayesian_network,False,statistical,NaN,ProbabilityEstimation
4,"ClosestDistance(L_2, accuracy)",50,100,0.48,1.08,0.0,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,bayesian_network,False,statistical,NaN,ClosestDistance
5,Groundhog,50,100,5495.68,1529.21,1.0,1.0,0.0,1.0,0.0,NaN,True,2.39,inf,2.21,inf,1.878,inf,privbayes,True,statistical,1.0,Groundhog
6,ShadowModelling(RandomQueries),50,100,70.01,1.03,1.0,1.0,0.0,1.0,0.0,NaN,True,2.39,inf,2.21,inf,1.878,inf,privbayes,True,statistical,1.0,ShadowModelling
7,"LocalNeighbourhood(L_2, 0.250411558405805, acc...",50,100,0.47,1.09,0.5,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,privbayes,True,statistical,1.0,LocalNeighbourhood
8,ProbabilityEstimation(KernelDensity()),50,100,0.56,0.94,0.0,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,privbayes,True,statistical,1.0,ProbabilityEstimation
9,"ClosestDistance(L_2, accuracy)",50,100,0.42,1.08,0.0,0.0,1.0,0.0,0.0,0.0,False,0.00,0.086,0.00,0.102,0.000,0.139,privbayes,True,statistical,1.0,ClosestDistance


**MIA AUC - attack x method:**

method,bayesian_network,privbayes,ctgan,dpgan
attack_short,,,,
Groundhog,1.0,1.0,1.0,1.0
ShadowModelling,1.0,1.0,1.0,1.0
LocalNeighbourhood,0.0,0.5,0.5,0.5
ProbabilityEstimation,0.0,0.0,1.0,1.0
ClosestDistance,0.0,0.0,0.0,0.0


**Membership advantage (tp - fp) - attack x method:**

method,bayesian_network,privbayes,ctgan,dpgan
attack_short,,,,
Groundhog,1.0,1.0,1.0,1.0
ShadowModelling,1.0,1.0,1.0,1.0
LocalNeighbourhood,0.0,0.0,0.0,0.0
ProbabilityEstimation,0.0,0.0,1.0,1.0
ClosestDistance,0.0,0.0,0.0,0.0


## 5. Computational cost  (from recorded overhead CSVs)

Generation wall-time from `sdg/computational_overhead.csv` (full 21,523-row synthesis); privacy-audit wall-time summed over the 5 attacks from `results/per_method/*/effeps_*.csv` (the first attack does all the generator retraining; the rest reuse cached fits). Utility + fidelity evaluation is negligible (<3 s per method). All runs on laptop CPU.

In [8]:
# Computational cost, assembled from the recorded overhead CSVs
BENCH = TABLES.parent.parent          # benchmark_tapas/
REPO  = BENCH.parent                  # repo root
PER_METHOD = TABLES.parent / "per_method"
sdg = pd.read_csv(REPO / "sdg" / "computational_overhead.csv")

def fmt(s):
    if s < 60:   return f"{s:.0f} s"
    if s < 3600: return f"{s/60:.1f} min"
    return f"{s/3600:.1f} h"

rows = []
for m, label in [("bayesian_network", "BN"), ("privbayes", "PrivBayes"),
                 ("ctgan", "CTGAN"), ("dpgan", "DPGAN")]:
    g = sdg[sdg.method == m].iloc[0]
    a = pd.read_csv(PER_METHOD / m / f"effeps_{m}.csv")
    rows.append({
        "SDG": label,
        "Generation (21,523 rows)": fmt(g.wall_time_s),
        "Gen peak (MB)": round(g.peak_memory_mb),
        "Privacy audit (5 attacks)": fmt(a.wall_time_s.sum()),
        "Audit peak (MB)": round(a.peak_memory_mb.max()),
    })

comp = pd.DataFrame(rows).set_index("SDG")
display(comp)
print("Utility + fidelity evaluation: negligible (<3 s per method).")

,"Generation (21,523 rows)",Gen peak (MB),Privacy audit (5 attacks),Audit peak (MB)
SDG,,,,
BN,13 s,44,4.3 min,359
PrivBayes,2.8 min,22,1.5 h,1529
CTGAN,1.1 h,65,24.6 min,328
DPGAN,1.9 h,134,2.1 h,322


Utility + fidelity evaluation: negligible (<3 s per method).
